# تصميم النظام واختيار النموذج (System Design & Model Selection) — DMorphNet

هذا الدفتر ينفّذ خطوة **"System Design and Model Selection Rationale"**: نظام من **مرحلتين** بدل نموذج واحد كبير.

## لماذا مرحلتان؟
- جرّبنا أولاً نموذج تعلّم عميق واحد (End-to-End)، لكن نتائجه على الصور الجديدة لم تكن جيدة — كان **يحفظ بيانات التدريب** (Overfitting) لأن مجموعة البيانات صغيرة نسبياً.
- الحل: فصل العمل إلى مرحلتين:

| المرحلة | الأداة | السبب |
|---|---|---|
| ١) استخراج السمات (Features) | **EfficientNet-B6** مدرّب مسبقاً على ImageNet | يرى التفاصيل الدقيقة في الوجه (آثار الدمج الصغيرة) دون تعقيد زائد |
| ٢) التصنيف | **SVM** | بسيط ويعمل ممتازاً مع البيانات القليلة، ويفصل بوضوح بين الحقيقي والمورف |

- جرّبنا أيضاً **MobileNet** و **ResNet** لاستخراج السمات، لكن نتائجهما أقل — وسنعيد هذه المقارنة عملياً في نهاية الدفتر.

> ⚙️ فعّل **GPU** قبل التشغيل: Runtime ← Change runtime type ← GPU. استخراج سمات 45,000 صورة بحجم 528×528 عبر B6 يستغرق ~١–٢ ساعة على T4.


## ١) تجهيز البيئة والتحقق من GPU

نستورد المكتبات ونتأكد من توفر GPU — بدونه سيكون استخراج السمات بطيئاً جداً.


In [ ]:
!pip -q install scikit-learn joblib tqdm

import os, glob, random, shutil
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

gpus = tf.config.list_physical_devices('GPU')
print("TensorFlow:", tf.__version__)
print("GPU المتاح:", gpus if gpus else "⚠️ لا يوجد GPU — فعّله من إعدادات الجلسة!")


## ٢) تحميل الصور المعالجة من الخطوة السابقة

نحمّل صور الـ 528×528 (بعد CLAHE والضغط) التي أنتجناها في دفتر **التوحيد والمعالجة المسبقة**:
- إن كانت موجودة في الجلسة استخدمناها مباشرة، وإلا فككنا الضغط من Google Drive.


In [ ]:
PROC_DIR = "/content/DMorphNet_processed"
PROC_ZIP = "/content/drive/MyDrive/DMorphNet_processed.zip"

if not os.path.isdir(os.path.join(PROC_DIR, "train")):
    from google.colab import drive
    drive.mount('/content/drive')
    print("جاري فك الضغط من Google Drive ...")
    shutil.unpack_archive(PROC_ZIP, PROC_DIR)

SPLITS = ["train", "val", "test"]
CLASSES = ["real", "morph"]          # 0 = حقيقية ، 1 = مدموجة
LABEL_MAP = {"real": 0, "morph": 1}

for split in SPLITS:
    counts = {c: len(glob.glob(os.path.join(PROC_DIR, split, c, "*.jpg")))
              for c in CLASSES}
    print(f"{split}: حقيقية={counts['real']}  مدموجة={counts['morph']}")


## ٣) الإعدادات العامة


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMG_SIZE = 528            # حجم إدخال EfficientNet-B6
BATCH_SIZE = 16           # دفعة استخراج السمات (قلّلها إذا امتلأت ذاكرة GPU)
AUTOTUNE = tf.data.AUTOTUNE

FEAT_DIR = "/content/DMorphNet_features"     # مجلد حفظ السمات المستخرجة
os.makedirs(FEAT_DIR, exist_ok=True)


def list_files(split):
    # إرجاع مسارات الصور وملصقاتها (0 حقيقية / 1 مدموجة)
    paths, labels = [], []
    for cls, lab in LABEL_MAP.items():
        fs = sorted(glob.glob(os.path.join(PROC_DIR, split, cls, "*.jpg")))
        paths += fs
        labels += [lab] * len(fs)
    return paths, np.array(labels, dtype=np.int32)

print("✅ الإعدادات جاهزة")


## ٤) المرحلة الأولى: بناء مستخرج السمات EfficientNet-B6

- نحمّل **EfficientNet-B6** بأوزان ImageNet **بدون طبقة التصنيف** (`include_top=False`).
- نضيف `pooling="avg"` (Global Average Pooling) فيصبح ناتج كل صورة **متجه سمات بطول 2304** يلخّص التفاصيل الدقيقة للوجه.
- النموذج **مجمّد** (لا تدريب) — نستخدمه كمستخرج سمات فقط، وهذا ما يمنع الحفظ الزائد على بياناتنا الصغيرة.


In [ ]:
from tensorflow.keras.applications import EfficientNetB6
from tensorflow.keras.applications.efficientnet import preprocess_input

extractor = EfficientNetB6(include_top=False, weights="imagenet",
                           pooling="avg", input_shape=(IMG_SIZE, IMG_SIZE, 3))
extractor.trainable = False        # مجمّد — استخراج سمات فقط

FEATURE_DIM = extractor.output_shape[-1]
print(f"✅ مستخرج السمات جاهز — طول متجه السمات لكل صورة: {FEATURE_DIM}")


def make_feature_ds(paths):
    # خط قراءة: فك JPEG ← float ← preprocess الخاص بـ EfficientNet ← دفعات
    ds = tf.data.Dataset.from_tensor_slices(paths)

    def load(p):
        img = tf.io.decode_jpeg(tf.io.read_file(p), channels=3)
        img = tf.cast(img, tf.float32)
        img = tf.ensure_shape(img, [IMG_SIZE, IMG_SIZE, 3])
        return preprocess_input(img)

    return ds.map(load, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)


## ٥) استخراج السمات لكل الأقسام وحفظها

نمرّر جميع الصور (تدريب/تحقق/اختبار — حقيقية ومدموجة بنفس الطريقة) عبر B6 ونحفظ السمات في ملفات `.npz` حتى لا نعيد الاستخراج كل مرة.

> ⏳ هذه أطول خلية (~١–٢ ساعة على GPU T4). إذا سبق أن حفظت السمات في Drive فسيتم تخطي الاستخراج تلقائياً.


In [ ]:
features, labels, paths_by_split = {}, {}, {}

for split in SPLITS:
    out_path = os.path.join(FEAT_DIR, f"effb6_{split}.npz")
    paths, y = list_files(split)
    paths_by_split[split] = paths

    if os.path.exists(out_path):
        data = np.load(out_path)
        features[split], labels[split] = data["X"], data["y"]
        print(f"{split}: تم تحميل السمات المحفوظة مسبقاً {features[split].shape}")
        continue

    print(f"استخراج سمات {split} ({len(paths)} صورة) ...")
    X = extractor.predict(make_feature_ds(paths), verbose=1)
    features[split], labels[split] = X, y
    np.savez_compressed(out_path, X=X, y=y)
    print(f"{split}: {X.shape} — تم الحفظ في {out_path}")

print("✅ تم استخراج سمات جميع الأقسام")


## ٦) المرحلة الثانية: تدريب مصنّف SVM

- نوحّد مقاييس السمات بـ `StandardScaler` (ضروري لأداء SVM).
- ندرّب **SVM بنواة RBF** على سمات التدريب (30,000 متجه) — بسيط وفعّال مع البيانات القليلة، ويرسم حداً فاصلاً واضحاً بين الوجوه الحقيقية والمدموجة.

> ⏳ تدريب RBF-SVM على 30k عيّنة قد يستغرق ٣٠–٩٠ دقيقة. للتجربة السريعة: ضع `TRAIN_SUBSET = 10000`، أو استخدم `LinearSVC` (أسرع بكثير وبدقة قريبة) بجعل `USE_LINEAR = True`.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, LinearSVC

TRAIN_SUBSET = None       # مثال: 10000 لتدريب أسرع على عيّنة جزئية، None = الكل
USE_LINEAR = False        # True = LinearSVC السريع بدل RBF

X_train, y_train = features["train"], labels["train"]
if TRAIN_SUBSET:
    idx = np.random.RandomState(SEED).permutation(len(X_train))[:TRAIN_SUBSET]
    X_train, y_train = X_train[idx], y_train[idx]

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)

if USE_LINEAR:
    svm = LinearSVC(C=1.0, random_state=SEED)
else:
    svm = SVC(kernel="rbf", C=10.0, gamma="scale", random_state=SEED)

print(f"تدريب SVM على {len(X_train_s)} عيّنة × {X_train_s.shape[1]} سمة ...")
svm.fit(X_train_s, y_train)
print("✅ اكتمل تدريب المصنّف")


## ٧) تقييم النظام على مجموعتي التحقق والاختبار

نقيس على بيانات **لم يرها النظام** (وهوياتها منفصلة تماماً عن التدريب):
- الدقة (Accuracy)، الضبط (Precision)، الاستدعاء (Recall)، ومقياس F1.
- **مصفوفة الالتباس** لمعرفة أين يخطئ النظام (حقيقية تُصنَّف مورف أو العكس).
- منحنى **ROC** ومساحة **AUC**.


In [ ]:
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_curve, auc)

results = {}
for split in ["val", "test"]:
    X_s = scaler.transform(features[split])
    y_true = labels[split]
    y_pred = svm.predict(X_s)
    y_score = svm.decision_function(X_s)
    results[split] = (y_true, y_pred, y_score)

    print(f"\n===== {split} =====")
    print("الدقة:", f"{accuracy_score(y_true, y_pred):.4f}")
    print(classification_report(y_true, y_pred,
                                target_names=["حقيقية (real)", "مدموجة (morph)"]))

# مصفوفة الالتباس + منحنى ROC لمجموعة الاختبار
y_true, y_pred, y_score = results["test"]
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm = confusion_matrix(y_true, y_pred)
im = axes[0].imshow(cm, cmap="Blues")
axes[0].set_title("مصفوفة الالتباس (اختبار)")
axes[0].set_xticks([0, 1]); axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(["حقيقية", "مدموجة"])
axes[0].set_yticklabels(["حقيقية", "مدموجة"])
axes[0].set_xlabel("التوقع"); axes[0].set_ylabel("الحقيقة")
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, cm[i, j], ha="center", va="center",
                     color="white" if cm[i, j] > cm.max() / 2 else "black")

fpr, tpr, _ = roc_curve(y_true, y_score)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, label=f"AUC = {roc_auc:.4f}")
axes[1].plot([0, 1], [0, 1], "--", color="gray")
axes[1].set_title("منحنى ROC (اختبار)")
axes[1].set_xlabel("معدل الإنذار الخاطئ"); axes[1].set_ylabel("معدل الكشف الصحيح")
axes[1].legend()
plt.tight_layout()
plt.show()


## ٨) مبرر الاختيار عملياً: مقارنة EfficientNet-B6 مع MobileNet و ResNet

نعيد التجربة المذكورة في الورقة: نفس المسار (سمات مجمّدة ← SVM) لكن بمستخرجات مختلفة:
- **MobileNetV2** (خفيف وسريع، إدخال 224×224).
- **ResNet50** (كلاسيكي، إدخال 224×224).
- **EfficientNet-B6** (سماتنا المستخرجة مسبقاً).

للتسريع نجري المقارنة على **عيّنة جزئية** (6,000 تدريب / 2,000 اختبار) وبمصنّف `LinearSVC` موحّد — المطلوب هو الترتيب النسبي بين النماذج لا الأرقام النهائية.


In [ ]:
from tensorflow.keras.applications import MobileNetV2, ResNet50
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mob_pre
from tensorflow.keras.applications.resnet import preprocess_input as res_pre

CMP_TRAIN, CMP_TEST, CMP_SIZE = 6000, 2000, 224

rng = np.random.RandomState(SEED)
tr_idx = rng.permutation(len(paths_by_split["train"]))[:CMP_TRAIN]
te_idx = rng.permutation(len(paths_by_split["test"]))[:CMP_TEST]

tr_paths = [paths_by_split["train"][i] for i in tr_idx]
te_paths = [paths_by_split["test"][i] for i in te_idx]
y_tr, y_te = labels["train"][tr_idx], labels["test"][te_idx]


def extract_with(model_cls, pre_fn, size, paths):
    # استخراج سمات بنموذج مجمّد آخر (لأغراض المقارنة)
    base = model_cls(include_top=False, weights="imagenet",
                     pooling="avg", input_shape=(size, size, 3))

    def load(p):
        img = tf.io.decode_jpeg(tf.io.read_file(p), channels=3)
        img = tf.image.resize(tf.cast(img, tf.float32), [size, size])
        return pre_fn(img)

    ds = (tf.data.Dataset.from_tensor_slices(paths)
          .map(load, num_parallel_calls=AUTOTUNE).batch(32).prefetch(AUTOTUNE))
    return base.predict(ds, verbose=1)


def quick_svm_acc(X_tr, X_te):
    # تدريب سريع موحّد للمقارنة العادلة بين المستخرجات
    sc = StandardScaler().fit(X_tr)
    clf = LinearSVC(C=1.0, random_state=SEED).fit(sc.transform(X_tr), y_tr)
    return accuracy_score(y_te, clf.predict(sc.transform(X_te)))


comparison = {}

print("--- MobileNetV2 ---")
comparison["MobileNetV2"] = quick_svm_acc(
    extract_with(MobileNetV2, mob_pre, CMP_SIZE, tr_paths),
    extract_with(MobileNetV2, mob_pre, CMP_SIZE, te_paths))

print("--- ResNet50 ---")
comparison["ResNet50"] = quick_svm_acc(
    extract_with(ResNet50, res_pre, CMP_SIZE, tr_paths),
    extract_with(ResNet50, res_pre, CMP_SIZE, te_paths))

print("--- EfficientNet-B6 (سمات محفوظة) ---")
comparison["EfficientNet-B6"] = quick_svm_acc(
    features["train"][tr_idx], features["test"][te_idx])

print("\nنتيجة المقارنة (دقة على عيّنة الاختبار):")
for name, acc in sorted(comparison.items(), key=lambda x: -x[1]):
    print(f"  {name:18s}: {acc:.4f}")
best = max(comparison, key=comparison.get)
print(f"\n🏆 الأفضل: {best} — وهذا يؤكد اختيارنا لـ EfficientNet-B6")


## ٩) حفظ النظام الكامل في Google Drive

نحفظ كل مكوّنات النظام لاستخدامها لاحقاً في التقييم أو النشر:
- مصنّف SVM المدرّب + مُوحِّد المقاييس (Scaler) بصيغة `joblib`.
- السمات المستخرجة (`.npz`) حتى لا نعيد المرور على B6 مرة أخرى.


In [ ]:
import joblib
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/DMorphNet_models"
os.makedirs(SAVE_DIR, exist_ok=True)

joblib.dump(svm, os.path.join(SAVE_DIR, "svm_classifier.joblib"))
joblib.dump(scaler, os.path.join(SAVE_DIR, "feature_scaler.joblib"))
for split in SPLITS:
    shutil.copy(os.path.join(FEAT_DIR, f"effb6_{split}.npz"), SAVE_DIR)

print("تم الحفظ في:", SAVE_DIR)
print(os.listdir(SAVE_DIR))
print("\n🎉 النظام كامل: EfficientNet-B6 (سمات) + SVM (تصنيف) — جاهز للتقييم والنشر")
